In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import yaml

from evaluate import evaluate_checkpoint

In [ ]:
def find_config(save_path):
    parent_path = f"{save_path}/../base_config.yaml"
    child_path = f"{save_path}/config.yaml"

    if os.path.exists(child_path):
        return child_path
    
    if os.path.exists(parent_path):
        return parent_path
    
    raise FileNotFoundError(f"Could not find config file in {save_path} or its parent directory.")

In [ ]:
class Result:
    def __init__(self, save_path: str):
        
        self.save_path = save_path

        self.config = yaml.safe_load(open(find_config(save_path), "r"))

        self.eval = np.load(f"{save_path}/eval/evaluations.npz")
        self.timesteps = self.eval["timesteps"].max()

        self.eval_result = None

    def evaluate_result_best(self):
        if self.eval_result is not None:
            return self.eval_result
        
        best_path = f"{self.save_path}/best/best_model.zip"
        config = find_config(self.save_path)
        eval_result = evaluate_checkpoint(best_path, config_override=config, n_episodes=50, seed=42)
        self.eval_result = eval_result
        return eval_result


In [ ]:
result_stages = [Result(f"results/ppo-tweaks/stage_{i}") for i in range(0, 5)]
baseline_result = Result("results/baseline-reallyforreal")

In [ ]:
bests = {
    "baseline": baseline_result.evaluate_result_best(),
    **{i: result.evaluate_result_best() for i, result in enumerate(result_stages)}
}

In [ ]:
baseline_best_idx = np.argmax(baseline_result.eval["successes"].mean(axis=1))
baseline_best_success = baseline_result.eval["successes"].mean(axis=1)[baseline_best_idx]
baseline_best_timestep = baseline_result.eval["timesteps"][baseline_best_idx]

print(f"Baseline best success rate: {baseline_best_success:.2f} at timestep {baseline_best_timestep}")

In [ ]:
for k, v in bests.items():
    print(k, v["success_rate"])

In [ ]:
def get_best_timestep(result):
    best_idx = np.argmax(result.eval["successes"].mean(axis=1))
    return result.eval["timesteps"][best_idx]

print("baseline", get_best_timestep(baseline_result))
total = 0
for i, result in enumerate(result_stages):
    timestep = get_best_timestep(result)
    total += timestep
    print(f"stage {i}", total)

In [ ]:
def plot_final_success(bests):
    keys = list(bests.keys())
    values = [bests[k]["success_rate"] for k in keys]
    keys = [str(k) for k in keys]
    keys = ["stage " + k if k != "baseline" else k for k in keys]
    colors = ["C0" if k == "baseline" else "C1" for k in keys]
    plt.bar(keys, values, color=colors)
    plt.ylim(0, 1)
    plt.ylabel("Robosuite Success Rate")
    plt.title("Best Checkpoint Final Robosuite Lift Task Success Rate")
    plt.xlabel("RL Training Run")
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.savefig("figures/final_success.png")
    plt.show()

plot_final_success(bests)

In [ ]:
def plot_final_success(baseline_result: Result, result_stages: list[Result]):
    baseline_timesteps = baseline_result.eval["timesteps"]
    baseline_successes = baseline_result.eval["successes"]

    total_timesteps = 0
    stages_timesteps = []
    stages_successes = []
    for result in result_stages:
        best_idx = np.argmax(result.eval["successes"].mean(axis=1))
        best_timestep = result.eval["timesteps"][best_idx]
        
        stages_timesteps.append(total_timesteps + best_timestep)
        stages_successes.append(result.eval_result["success_rate"])

        total_timesteps += result.timesteps

    baseline_timesteps = [step for step in baseline_timesteps if step <= 1e7]
    baseline_successes = baseline_successes[:len(baseline_timesteps)]

    plt.plot(baseline_timesteps, baseline_successes.mean(axis=1), label="Baseline")
    plt.plot(stages_timesteps, stages_successes, marker="o", label="Curriculum")
    plt.title("Success Rate on Final Robosuite Lift Task")
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.legend()
    plt.ylabel("Robosuite Success Rate")
    plt.xlabel("Timesteps")
    plt.savefig("figures/success_over_time.png")
    plt.show()

plot_final_success(baseline_result, result_stages)

In [ ]:
def plot_stages_success_rate(result_stages: list[Result]):
    total_timesteps = 0
    for i, result in enumerate(result_stages):
        timesteps = result.eval["timesteps"]
        success = result.eval["successes"].mean(axis=1)

        if total_timesteps > 0:
            plt.axvline(total_timesteps, color="gray", linestyle="--", alpha=0.5)
        plt.plot(timesteps + total_timesteps, success, color="C1")

        total_timesteps += result.timesteps

    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.axhline(0.9, color="red", linestyle="--", label="Success Threshold")
    plt.axhline(0.75, color="orange", linestyle="--", label="Moderate Success Threshold")
    plt.legend()
    plt.title("Stage Success Rate During Training")
    plt.ylabel("Stage Success Rate")
    plt.xlabel("Timesteps")
    plt.savefig("figures/stages.png")
    plt.show()

plot_stages_success_rate(result_stages)